In [ ]:
from pyspark.sql import SparkSession


In [4]:
from pyspark.sql.functions import col, count, when,sum, date_diff,to_date

from pyspark.sql.types import DecimalType, IntegerType, StringType, DateType


In [6]:
spark = SparkSession.builder.appName('Project- Real world dataset').master('yarn')\
.getOrCreate()

25/10/01 05:06:54 INFO SparkEnv: Registering MapOutputTracker
25/10/01 05:06:54 INFO SparkEnv: Registering BlockManagerMaster
25/10/01 05:06:54 INFO SparkEnv: Registering BlockManagerMasterHeartbeat
25/10/01 05:06:54 INFO SparkEnv: Registering OutputCommitCoordinator


## Module 1 - Understaing data

In [3]:
hdfs_path= '/user/ankiitsharma09/data/'

In [4]:
customer_df= spark.read\
.format('csv')\
.option('header', 'true')\
.option('inferSchema', 'true')\
.load(hdfs_path+'olist_customers_dataset.csv')

In [5]:
customer_df.show(5)

+--------------------+--------------------+------------------------+--------------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state|
+--------------------+--------------------+------------------------+--------------------+--------------+
|06b8999e2fba1a1fb...|861eff4711a542e4b...|                   14409|              franca|            SP|
|18955e83d337fd6b2...|290c77bc529b7ac93...|                    9790|sao bernardo do c...|            SP|
|4e7b3e00288586ebd...|060e732b5b29e8181...|                    1151|           sao paulo|            SP|
|b2b6027bc5c5109e5...|259dac757896d24d7...|                    8775|     mogi das cruzes|            SP|
|4f2d8ab171c80ec83...|345ecd01c38d18a90...|                   13056|            campinas|            SP|
+--------------------+--------------------+------------------------+--------------------+--------------+
only showing top 5 rows



In [6]:
duplicate_record= customer_df.groupBy('customer_id').count().filter('count>1').show()

+-----------+-----+
|customer_id|count|
+-----------+-----+
+-----------+-----+



In [7]:


check_null = customer_df.select([
    count(when(col(c).isNull(), 1)).alias(c) 
    for c in customer_df.columns
]).show()

+-----------+------------------+------------------------+-------------+--------------+
|customer_id|customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|
+-----------+------------------+------------------------+-------------+--------------+
|          0|                 0|                       0|            0|             0|
+-----------+------------------+------------------------+-------------+--------------+



In [8]:
#customer_distribution by state

customer_df.groupBy('customer_state').count()\
.orderBy('count' , ascending= False).show()

+--------------+-----+
|customer_state|count|
+--------------+-----+
|            SP|41746|
|            RJ|12852|
|            MG|11635|
|            RS| 5466|
|            PR| 5045|
|            SC| 3637|
|            BA| 3380|
|            DF| 2140|
|            ES| 2033|
|            GO| 2020|
|            PE| 1652|
|            CE| 1336|
|            PA|  975|
|            MT|  907|
|            MA|  747|
|            MS|  715|
|            PB|  536|
|            PI|  495|
|            RN|  485|
|            AL|  413|
+--------------+-----+
only showing top 20 rows



In [9]:
orders_df= spark.read\
.format('csv')\
.option('header', 'true')\
.option('inferSchema', 'true')\
.load(hdfs_path+'olist_orders_dataset.csv')


In [10]:
print(f'Customers : {customer_df.count()} rows')
print(f'Orders : {orders_df.count()} rows')


Customers : 99441 rows
Orders : 99441 rows


In [11]:
orders_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



In [12]:
duplicate_record= orders_df.groupBy('order_id').count().filter('count>1').show()

+--------+-----+
|order_id|count|
+--------+-----+
+--------+-----+



In [13]:
orders_df.select([count(when(col(c).isNull(),1)).alias(c) for c in orders_df.columns]).show()



+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|       0|          0|           0|                       0|              160|                        1783|                         2965|                            0|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+



In [14]:
orders_df.groupBy('order_status').count().orderBy('count' , ascending=False).show()

+------------+-----+
|order_status|count|
+------------+-----+
|   delivered|96478|
|     shipped| 1107|
|    canceled|  625|
| unavailable|  609|
|    invoiced|  314|
|  processing|  301|
|     created|    5|
|    approved|    2|
+------------+-----+



In [15]:
order_items_df= spark.read\
.format('csv')\
.option('header', 'true')\
.option('inferSchema', 'true')\
.load(hdfs_path+'olist_order_items_dataset.csv')

In [16]:
geolocation_df= spark.read\
.format('csv')\
.option('header', 'true')\
.option('inferSchema', 'true')\
.load(hdfs_path+'olist_geolocation_dataset.csv')

order_payments_df= spark.read\
.format('csv')\
.option('header', 'true')\
.option('inferSchema', 'true')\
.load(hdfs_path+'olist_order_payments_dataset.csv')

order_reviews_df= spark.read\
.format('csv')\
.option('header', 'true')\
.option('inferSchema', 'true')\
.load(hdfs_path+'olist_order_reviews_dataset.csv')

products_df= spark.read\
.format('csv')\
.option('header', 'true')\
.option('inferSchema', 'true')\
.load(hdfs_path+'olist_products_dataset.csv')

sellers_df= spark.read\
.format('csv')\
.option('header', 'true')\
.option('inferSchema', 'true')\
.load(hdfs_path+'olist_sellers_dataset.csv')



In [17]:
order_items_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)



In [18]:
top_products= order_items_df.groupBy('product_id').agg(sum('price').alias('total_sales'))

In [19]:
top_products.orderBy('total_sales', ascending=False).show()

+--------------------+------------------+
|          product_id|       total_sales|
+--------------------+------------------+
|bb50f2e236e5eea01...|           63885.0|
|6cdd53843498f9289...| 54730.20000000005|
|d6160fb7873f18409...|48899.340000000004|
|d1c427060a0f73f6b...| 47214.51000000006|
|99a4788cb24856965...|43025.560000000085|
|3dd2a17168ec895c7...| 41082.60000000005|
|25c38557cf793876c...| 38907.32000000001|
|5f504b3a1c75b73d6...|37733.899999999994|
|53b36df67ebb7c415...| 37683.42000000001|
|aca2eb7d00ea1a7b8...| 37608.90000000007|
|e0d64dcfaa3b6db5c...|          31786.82|
|d285360f29ac7fd97...|31623.809999999983|
|7a10781637204d8d1...|           30467.5|
|f1c7f353075ce59d8...|          29997.36|
|f819f0c84a64f02d3...|29024.479999999996|
|588531f8ec37e7d5f...|28291.989999999998|
|422879e10f4668299...|26577.219999999972|
|16c4e87b98a9370a9...|           25034.0|
|5a848e4ab52fd5445...|24229.029999999962|
|a62e25e09e05e6faf...|           24051.0|
+--------------------+------------

In [20]:
orders_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



In [21]:
delivery_df = orders_df.select('order_id','order_delivered_customer_date','order_purchase_timestamp')

In [22]:
delivery_delivery_time_df= delivery_df.withColumn('delivered_time', date_diff(col('order_delivered_customer_date'), col('order_purchase_timestamp')))

In [23]:
delivery_delivery_time_df.orderBy('delivered_time', ascending= False).show()

+--------------------+-----------------------------+------------------------+--------------+
|            order_id|order_delivered_customer_date|order_purchase_timestamp|delivered_time|
+--------------------+-----------------------------+------------------------+--------------+
|ca07593549f1816d2...|          2017-09-19 14:36:39|     2017-02-21 23:31:27|           210|
|1b3190b2dfa9d789e...|          2018-09-19 23:24:07|     2018-02-23 14:57:35|           208|
|440d0d17af552815d...|          2017-09-19 15:12:50|     2017-03-07 23:59:51|           196|
|2fb597c2f772eca01...|          2017-09-19 14:33:17|     2017-03-08 18:09:02|           195|
|285ab9426d6982034...|          2017-09-19 14:00:04|     2017-03-08 22:47:40|           195|
|0f4519c5f1c541dde...|          2017-09-19 14:38:21|     2017-03-09 13:26:57|           194|
|47b40429ed8cce3ae...|          2018-07-13 20:51:31|     2018-01-03 09:44:01|           191|
|2fe324febf907e3ea...|          2017-09-19 17:00:07|     2017-03-13 20

## Module 2- data cleaning 

In [24]:
!hadoop fs -ls /user/ankiitsharma09/data/

Found 9 items
-rw-r--r--   2 ankiitsharma09 hadoop    9033957 2025-09-27 02:56 /user/ankiitsharma09/data/olist_customers_dataset.csv
-rw-r--r--   2 ankiitsharma09 hadoop   61273883 2025-09-27 02:56 /user/ankiitsharma09/data/olist_geolocation_dataset.csv
-rw-r--r--   2 ankiitsharma09 hadoop   15438671 2025-09-27 02:56 /user/ankiitsharma09/data/olist_order_items_dataset.csv
-rw-r--r--   2 ankiitsharma09 hadoop    5777138 2025-09-27 02:56 /user/ankiitsharma09/data/olist_order_payments_dataset.csv
-rw-r--r--   2 ankiitsharma09 hadoop   14451670 2025-09-27 02:56 /user/ankiitsharma09/data/olist_order_reviews_dataset.csv
-rw-r--r--   2 ankiitsharma09 hadoop   17654914 2025-09-27 02:56 /user/ankiitsharma09/data/olist_orders_dataset.csv
-rw-r--r--   2 ankiitsharma09 hadoop    2379446 2025-09-27 02:56 /user/ankiitsharma09/data/olist_products_dataset.csv
-rw-r--r--   2 ankiitsharma09 hadoop     174703 2025-09-27 02:56 /user/ankiitsharma09/data/olist_sellers_dataset.csv
-rw-r--r--   2 ankiitsharma

In [25]:
def load_data(file_name, alias=None):
    """
    Load CSV from HDFS and return a Spark DataFrame.
    Args:
        file_name (str): CSV file name in HDFS path.
        alias (str, optional): Alias name for printing/logging.
    
    Returns:
        DataFrame: Spark DataFrame loaded from the given file.
    """
    hdfs_path = f'/user/ankiitsharma09/data/{file_name}'
    
    df = spark.read \
        .format('csv') \
        .option('header', 'true') \
        .option('inferSchema', 'true') \
        .load(hdfs_path)
    
    print(f"DataFrame created for {alias if alias else file_name}")
    return df


In [26]:
customers_df= load_data("olist_customers_dataset.csv","customers_df")
geolocation_df= load_data("olist_geolocation_dataset.csv")
order_items_df= load_data("olist_order_items_dataset.csv")
order_payments_df= load_data("olist_order_payments_dataset.csv")
order_reviews_df= load_data("olist_order_reviews_dataset.csv")
orders_df= load_data("olist_orders_dataset.csv")
products_df= load_data("olist_products_dataset.csv")
sellers_df= load_data("olist_sellers_dataset.csv")
product_category_df= load_data("product_category_name_translation.csv")
    

DataFrame created for customers_df


DataFrame created for olist_geolocation_dataset.csv
DataFrame created for olist_order_items_dataset.csv
DataFrame created for olist_order_payments_dataset.csv
DataFrame created for olist_order_reviews_dataset.csv
DataFrame created for olist_orders_dataset.csv
DataFrame created for olist_products_dataset.csv
DataFrame created for olist_sellers_dataset.csv
DataFrame created for product_category_name_translation.csv


In [27]:
def missing_values(df):
    print(f"Missing values for {df}")
    df.select([count(when(col(c).isNull(),1)).alias(c) for c in orders_df.columns]).show()


    
    

In [28]:
from pyspark.sql.functions import col, when, count

def missing_values(df, alias="DataFrame"):
    """
    Print missing values count for each column in a DataFrame.
    
    Args:
        df (DataFrame): Spark DataFrame.
        alias (str): Optional name/alias for display.
    """
    print(f"Missing values for {alias}:")
    df.select([
        count(when(col(c).isNull(), 1)).alias(c) for c in df.columns
    ]).show()

In [29]:
missing_values(customers_df,"customers_df")
missing_values(geolocation_df,"geolocation_df")
missing_values(order_items_df,"order_items_df")
missing_values(order_payments_df,"order_payments_df")
missing_values(order_reviews_df,"order_reviews_df")
missing_values(orders_df,"orders_df")
missing_values(products_df,"products_df")
missing_values(sellers_df,"sellers_df")
missing_values(product_category_df,"product_category_df")

Missing values for customers_df:
+-----------+------------------+------------------------+-------------+--------------+
|customer_id|customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|
+-----------+------------------+------------------------+-------------+--------------+
|          0|                 0|                       0|            0|             0|
+-----------+------------------+------------------------+-------------+--------------+

Missing values for geolocation_df:


+---------------------------+---------------+---------------+----------------+-----------------+
|geolocation_zip_code_prefix|geolocation_lat|geolocation_lng|geolocation_city|geolocation_state|
+---------------------------+---------------+---------------+----------------+-----------------+
|                          0|              0|              0|               0|                0|
+---------------------------+---------------+---------------+----------------+-----------------+

Missing values for order_items_df:
+--------+-------------+----------+---------+-------------------+-----+-------------+
|order_id|order_item_id|product_id|seller_id|shipping_limit_date|price|freight_value|
+--------+-------------+----------+---------+-------------------+-----+-------------+
|       0|            0|         0|        0|                  0|    0|            0|
+--------+-------------+----------+---------+-------------------+-----+-------------+

Missing values for order_payments_df:
+--------+

+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+
|review_id|order_id|review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+
|        1|    2236|        2380|               92157|                 63079|                8764|                   8785|
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+

Missing values for orders_df:
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------+--------

In [30]:
#Dealing with missing values

In [31]:
#Drop null values

orders_df_cleaned= orders_df.na.drop(subset=['customer_id','customer_id','order_status'])

In [32]:
missing_values(orders_df_cleaned,"orders_df")


Missing values for orders_df:
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|       0|          0|           0|                       0|              160|                        1783|                         2965|                            0|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+



In [33]:
#filling null values with a timestamp as type is timestamp

from pyspark.sql.functions import when, lit, to_timestamp, col

orders_df_cleaned = orders_df.withColumn(
    "order_approved_at",
    when(
        col("order_approved_at").isNull(),
        to_timestamp(lit("9999-12-31 00:00:00"))  # fill value
    ).otherwise(col("order_approved_at"))
)


In [34]:
missing_values(orders_df_cleaned,"order_df_cleaned")


Missing values for order_df_cleaned:
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|       0|          0|           0|                       0|                0|                        1783|                         2965|                            0|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+



In [35]:
#compare the null values with total no of rows
orders_df_cleaned.selectExpr(
    "count(*) as total_rows",
    "count(order_approved_at) as non_nulls",
    "count(*) - count(order_approved_at) as still_nulls"
).show()

+----------+---------+-----------+
|total_rows|non_nulls|still_nulls|
+----------+---------+-----------+
|     99441|    99441|          0|
+----------+---------+-----------+



In [36]:
#imput missing values 

from pyspark.ml.feature import Imputer

In [37]:
imputer= Imputer(inputCols=['payment_value'], outputCols=['payment_value-Imputed']).setStrategy('mean')

In [38]:
imputer= Imputer(inputCols=['payment_value'], outputCols=['payment_value-Imputed']).setStrategy('mean')
payment_df_cleaned= imputer.fit(order_payments_df).transform(order_payments_df)


In [39]:
payment_df_cleaned.show(5)

+--------------------+------------------+------------+--------------------+-------------+---------------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|payment_value-Imputed|
+--------------------+------------------+------------+--------------------+-------------+---------------------+
|b81ef226f3fe1789b...|                 1| credit_card|                   8|        99.33|                99.33|
|a9810da82917af2d9...|                 1| credit_card|                   1|        24.39|                24.39|
|25e8ea4e93396b6fa...|                 1| credit_card|                   1|        65.71|                65.71|
|ba78997921bbcdc13...|                 1| credit_card|                   8|       107.78|               107.78|
|42fdf880ba16b47b5...|                 1| credit_card|                   2|       128.45|               128.45|
+--------------------+------------------+------------+--------------------+-------------+---------------

In [40]:

payment_df_cleaned_with_null= order_payments_df.withColumn("payment_value1", when(col('payment_value').isNotNull(), col('payment_value')).otherwise("Null"))

In [41]:
payment_df_cleaned_with_null.show()

+--------------------+------------------+------------+--------------------+-------------+--------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|payment_value1|
+--------------------+------------------+------------+--------------------+-------------+--------------+
|b81ef226f3fe1789b...|                 1| credit_card|                   8|        99.33|         99.33|
|a9810da82917af2d9...|                 1| credit_card|                   1|        24.39|         24.39|
|25e8ea4e93396b6fa...|                 1| credit_card|                   1|        65.71|         65.71|
|ba78997921bbcdc13...|                 1| credit_card|                   8|       107.78|        107.78|
|42fdf880ba16b47b5...|                 1| credit_card|                   2|       128.45|        128.45|
|298fcdf1f73eb413e...|                 1| credit_card|                   2|        96.12|         96.12|
|771ee386b001f0620...|                 1| credit_card| 

In [42]:

#Standardize the format

def print_schema(df,df_name):
    print(df_name)
    df.printSchema()
    

In [43]:
print_schema(customers_df,"customers_df")
print_schema(geolocation_df,"geolocation_df")
print_schema(order_items_df,"order_items_df")
print_schema(order_payments_df,"order_payments_df")
print_schema(order_reviews_df,"order_reviews_df")
print_schema(orders_df,"orders_df")
print_schema(products_df,"products_df")
print_schema(sellers_df,"sellers_df")
print_schema(product_category_df,"product_category_df")

customers_df
root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)

geolocation_df
root
 |-- geolocation_zip_code_prefix: integer (nullable = true)
 |-- geolocation_lat: double (nullable = true)
 |-- geolocation_lng: double (nullable = true)
 |-- geolocation_city: string (nullable = true)
 |-- geolocation_state: string (nullable = true)

order_items_df
root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)

order_payments_df
root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (n

In [44]:
'''
orders_df
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)'''


order_df_cleaned= orders_df.withColumn('order_purchase_timestamp', to_date(col('order_purchase_timestamp')))

In [45]:
order_payments_df.groupBy('payment_type').count().show()

+------------+-----+
|payment_type|count|
+------------+-----+
| not_defined|    3|
| credit_card|76795|
|      boleto|19784|
|  debit_card| 1529|
|     voucher| 5775|
+------------+-----+



In [46]:
payment_df_cleaned= order_payments_df.withColumn('payment_type', when(col('payment_type')=='boleto','Bank Transfer')\
.when(col('payment_type')=='credit_card','Credit Card')\
.when(col('payment_type')=='not_defined','Unknown')\
.when(col('payment_type')=='voucher','Voucher')\
.otherwise(col("payment_type"))) 



In [47]:
payment_df_cleaned.groupBy('payment_type').count().orderBy('count', ascending=False).show()

+-------------+-----+
| payment_type|count|
+-------------+-----+
|  Credit Card|76795|
|Bank Transfer|19784|
|      Voucher| 5775|
|   debit_card| 1529|
|      Unknown|    3|
+-------------+-----+



In [48]:
print_schema(customers_df,"customers_df")

customers_df
root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



In [49]:
#change dataType if required 

customer_df_cleaned= customers_df.withColumn('customer_zip_code_prefix', col('customer_zip_code_prefix').cast('string'))

In [50]:
print_schema(customer_df_cleaned,"customer_df_cleaned")

customer_df_cleaned
root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



In [51]:
#remove duplicates 

customer_df_cleaned= customers_df.dropDuplicates(['customer_id'])

In [52]:
customer_df_cleaned.columns

['customer_id',
 'customer_unique_id',
 'customer_zip_code_prefix',
 'customer_city',
 'customer_state']

In [53]:
customer_df_cleaned.describe().show()

25/09/30 06:51:20 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+--------------------+--------------------+------------------------+-------------------+--------------+
|summary|         customer_id|  customer_unique_id|customer_zip_code_prefix|      customer_city|customer_state|
+-------+--------------------+--------------------+------------------------+-------------------+--------------+
|  count|               99441|               99441|                   99441|              99441|         99441|
|   mean|                NULL|                NULL|       35137.47458291851|               NULL|          NULL|
| stddev|                NULL|                NULL|      29797.938996206147|               NULL|          NULL|
|    min|00012a2ce6f8dcda2...|0000366f3b9a7992b...|                    1003|abadia dos dourados|            AC|
|    max|ffffe8b65bbe3087b...|ffffd2657e2aad290...|                   99990|             zortea|            TO|
+-------+--------------------+--------------------+------------------------+-------------------+--------

In [54]:
#Join dataFrame

order_with_details= order_df_cleaned.join(order_items_df, 'order_id','left')\
.join(payment_df_cleaned, 'order_id','left')\
.join(customer_df_cleaned, 'customer_id', 'left')



In [55]:
order_with_details.columns

['customer_id',
 'order_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date',
 'order_item_id',
 'product_id',
 'seller_id',
 'shipping_limit_date',
 'price',
 'freight_value',
 'payment_sequential',
 'payment_type',
 'payment_installments',
 'payment_value',
 'customer_unique_id',
 'customer_zip_code_prefix',
 'customer_city',
 'customer_state']

In [56]:
order_with_details.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: date (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: double (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- c

In [57]:
order_with_details.show(5)

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+------------------+-------------+--------------------+-------------+--------------------+------------------------+-------------+--------------+
|         customer_id|            order_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|order_item_id|          product_id|           seller_id|shipping_limit_date|price|freight_value|payment_sequential| payment_type|payment_installments|payment_value|  customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+----------

In [58]:
#Feature engineering 

order_with_total_value= order_with_details.groupBy('order_id').agg(sum('payment_value').alias('Total_order_value'))

In [59]:
order_with_total_value.orderBy('Total_order_value', ascending=False).show()

+--------------------+------------------+
|            order_id| Total_order_value|
+--------------------+------------------+
|03caa2c082116e1d3...|         109312.64|
|ab14fdcfbe524636d...| 45256.00000000001|
|1b15974a0141d54e3...|44048.000000000015|
|2cc9089445046817a...|          36489.24|
|e8fa22c3673b1dd17...|30185.999999999993|
|736e1922ae60d0d6a...|          29099.52|
|9aec4e1ae90b23c7b...|           22346.6|
|71dab1155600756af...|21874.049999999996|
|912343626f370ead5...|          19457.04|
|4412d97cb2093633a...|          19174.38|
|9f5054bd9a3c71702...|18667.000000000004|
|428a2f660dc84138d...|          18384.75|
|3a213fcdfe7d98be7...|          17786.88|
|f80549a97eb203e15...|           17671.0|
|cf4659487be50c0c3...|          17069.76|
|be382a9e1ed251281...|16313.600000000002|
|637617b3ffe9e2f7a...|14963.639999999998|
|c52c7fbe316b5b9d5...|14577.569999999998|
|f60ce04ff8060152c...|14401.000000000002|
|73c8ab38f07dc9438...|14196.280000000004|
+--------------------+------------

In [60]:
order_with_total_value.withColumn('Total_order_value', col('Total_order_value').cast('integer')).orderBy('Total_order_value', ascending=False).show(5)

+--------------------+-----------------+
|            order_id|Total_order_value|
+--------------------+-----------------+
|03caa2c082116e1d3...|           109312|
|ab14fdcfbe524636d...|            45256|
|1b15974a0141d54e3...|            44048|
|2cc9089445046817a...|            36489|
|e8fa22c3673b1dd17...|            30185|
+--------------------+-----------------+
only showing top 5 rows



In [61]:
#delivery time calculate

order_with_details.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: date (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: double (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- c

In [62]:
order_with_details_Df= order_with_details.withColumn('order_delivery_Time', date_diff(col('order_delivered_customer_date'),col('order_purchase_timestamp')))

In [63]:
order_with_details_Df.select('order_id','order_delivered_customer_date','order_purchase_timestamp','order_delivery_Time').show()

+--------------------+-----------------------------+------------------------+-------------------+
|            order_id|order_delivered_customer_date|order_purchase_timestamp|order_delivery_Time|
+--------------------+-----------------------------+------------------------+-------------------+
|e481f51cbdc54678b...|          2017-10-10 21:25:13|              2017-10-02|                  8|
|e481f51cbdc54678b...|          2017-10-10 21:25:13|              2017-10-02|                  8|
|e481f51cbdc54678b...|          2017-10-10 21:25:13|              2017-10-02|                  8|
|53cdb2fc8bc7dce0b...|          2018-08-07 15:27:45|              2018-07-24|                 14|
|47770eb9100c2d0c4...|          2018-08-17 18:06:29|              2018-08-08|                  9|
|949d5b44dbf5de918...|          2017-12-02 00:28:42|              2017-11-18|                 14|
|ad21c59c0840e6cb8...|          2018-02-16 18:17:02|              2018-02-13|                  3|
|a4591c265e18cb1dc..

In [64]:

imputer= Imputer(inputCols=['order_delivery_Time'], outputCols=['order_delivery_Time-Imputed']).setStrategy('median')
order_with_details_Df_cleaned= imputer.fit(order_with_details_Df).transform(order_with_details_Df)


In [65]:
order_with_details_Df_cleaned.select('order_id','order_delivered_customer_date','order_purchase_timestamp','order_delivery_Time', 'order_delivery_Time-Imputed').show()

+--------------------+-----------------------------+------------------------+-------------------+---------------------------+
|            order_id|order_delivered_customer_date|order_purchase_timestamp|order_delivery_Time|order_delivery_Time-Imputed|
+--------------------+-----------------------------+------------------------+-------------------+---------------------------+
|e481f51cbdc54678b...|          2017-10-10 21:25:13|              2017-10-02|                  8|                          8|
|e481f51cbdc54678b...|          2017-10-10 21:25:13|              2017-10-02|                  8|                          8|
|e481f51cbdc54678b...|          2017-10-10 21:25:13|              2017-10-02|                  8|                          8|
|53cdb2fc8bc7dce0b...|          2018-08-07 15:27:45|              2018-07-24|                 14|                         14|
|47770eb9100c2d0c4...|          2018-08-17 18:06:29|              2018-08-08|                  9|                     

In [66]:
order_items_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)



In [67]:
#advance transformation

quantiles= order_items_df.approxQuantile('price',[0.001,0.99],0.0)

In [68]:
low_cutOff, high_cutOff= quantiles[0],quantiles[1]

In [69]:
order_items_df.select('price').summary().show()

+-------+------------------+
|summary|             price|
+-------+------------------+
|  count|            112650|
|   mean|120.65373901471354|
| stddev|183.63392805026012|
|    min|              0.85|
|    25%|              39.9|
|    50%|             74.99|
|    75%|             134.9|
|    max|            6735.0|
+-------+------------------+



In [70]:
order_item_cleaned= order_items_df.filter((col('price')>=low_cutOff) & (col('price')<=high_cutOff))

In [71]:
order_item_cleaned.show(5)

+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date|price|freight_value|
+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|00010242fe8c5a6d1...|            1|4244733e06e7ecb49...|48436dade18ac8b2b...|2017-09-19 09:45:35| 58.9|        13.29|
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|2017-05-03 11:05:13|239.9|        19.93|
|000229ec398224ef6...|            1|c777355d18b72b67a...|5b51032eddd242adc...|2018-01-18 14:48:30|199.0|        17.87|
|00024acbcdf0a6daa...|            1|7634da152a4610f15...|9d7a1d34a50524090...|2018-08-15 10:10:18|12.99|        12.79|
|00042b26cf59d7ce6...|            1|ac6c3623068f30de0...|df560393f3a51e745...|2017-02-13 13:57:51|199.9|        18.14|
+--------------------+-------------+------------

In [72]:
products_df.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)



In [73]:
product_df_cleaned= products_df.withColumn('Product_size_category', 
                                           when(col('product_weight_g')<500, 'Small')\
                                           .when(col('product_weight_g').between(500,2000),'Medium')\
                                           .otherwise('Large')
                                          )

In [74]:
product_df_cleaned.select('Product_size_category').show()

+---------------------+
|Product_size_category|
+---------------------+
|                Small|
|               Medium|
|                Small|
|                Small|
|               Medium|
|                Small|
|                Large|
|               Medium|
|                Small|
|               Medium|
|               Medium|
|                Large|
|                Small|
|               Medium|
|                Small|
|               Medium|
|                Small|
|               Medium|
|               Medium|
|               Medium|
+---------------------+
only showing top 20 rows



In [76]:
order_payments_df.show()

+--------------------+------------------+------------+--------------------+-------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------------------+------------------+------------+--------------------+-------------+
|b81ef226f3fe1789b...|                 1| credit_card|                   8|        99.33|
|a9810da82917af2d9...|                 1| credit_card|                   1|        24.39|
|25e8ea4e93396b6fa...|                 1| credit_card|                   1|        65.71|
|ba78997921bbcdc13...|                 1| credit_card|                   8|       107.78|
|42fdf880ba16b47b5...|                 1| credit_card|                   2|       128.45|
|298fcdf1f73eb413e...|                 1| credit_card|                   2|        96.12|
|771ee386b001f0620...|                 1| credit_card|                   1|        81.16|
|3d7239c394a212faa...|                 1| credit_card|                   3|        51.84|
|1f78449c8

In [ ]:
order_items_df.printSchema()

In [ ]:
from pyspark.sql.functions import sum as _sum

revenue_per_seller = order_order_payments_details.groupBy("seller_id") \
    .agg(_sum("payment_value").alias("total_revenue")) \
    .orderBy("total_revenue", ascending=False)

revenue_per_seller.show()

In [ ]:
#save as Sql table

revenue_per_seller.write.mode("overwrite").saveAsTable("default.revenue_per_seller")


In [ ]:
spark.sql("SHOW TABLES IN default").show()


In [ ]:
spark.sql("DESCRIBE default.revenue_per_seller").show()


In [ ]:
spark.sql("SELECT * FROM default.revenue_per_seller LIMIT 10").show()


In [ ]:
!hadoop fs -ls  /user/

In [ ]:
revenue_per_seller.write.mode('overwrite').parquet('/user/data_OutPut/revenue_per_seller')


## Module 3- data integration and agg

1- Joining dataset efficiently
2- optimizing joins
3- performaing aggrations
4- caching and optimizing the query for performance 

In [ ]:
spark.stop()


In [5]:
from pyspark.sql.functions import col, count, when,sum as _sum, round as _round, date_diff,to_date,lit,countDistinct,avg,stddev,broadcast,desc,rank,dense_rank, row_number
from pyspark.sql import Window


In [10]:
spark= SparkSession.builder.appName('DataIntegration').master('yarn').enableHiveSupport().getOrCreate()

25/10/01 05:07:55 INFO SparkEnv: Registering MapOutputTracker
25/10/01 05:07:55 INFO SparkEnv: Registering BlockManagerMaster
25/10/01 05:07:55 INFO SparkEnv: Registering BlockManagerMasterHeartbeat
25/10/01 05:07:55 INFO SparkEnv: Registering OutputCommitCoordinator


In [ ]:
!hadoop fs -ls /user/ankiitsharma09/data/

In [6]:
def loadData(file_name,df_name=None):
    hdfs_path=f'/user/ankiitsharma09/data/{file_name}'
    print('data load for : ', df_name)
    return spark.read.csv(hdfs_path, header=True, inferSchema=True)


In [7]:
customer_df= loadData('olist_customers_dataset.csv', 'customer_df')
geolocation_df= loadData('olist_geolocation_dataset.csv', 'geolocation_df')
order_items_df= loadData('olist_order_items_dataset.csv', 'order_items_df')
order_payments_df= loadData('olist_order_payments_dataset.csv', 'order_payments_df')
order_reviews_df= loadData('olist_order_reviews_dataset.csv', 'order_reviews_df')
orders_df= loadData('olist_orders_dataset.csv', 'orders_df')
products_df= loadData('olist_products_dataset.csv', 'products_df')
sellers_df= loadData('olist_sellers_dataset.csv', 'sellers_df')
product_category_name_df= loadData('product_category_name_translation.csv', 'product_category_name_df')

data load for :  customer_df


data load for :  geolocation_df


data load for :  order_items_df


data load for :  order_payments_df
data load for :  order_reviews_df
data load for :  orders_df
data load for :  products_df
data load for :  sellers_df
data load for :  product_category_name_df


In [8]:
#cache frequently used data

customer_df.cache()
order_items_df.cache()
order_payments_df.cache()
orders_df.cache()
order_reviews_df.cache()
products_df.cache()
sellers_df.cache()


DataFrame[seller_id: string, seller_zip_code_prefix: int, seller_city: string, seller_state: string]

In [20]:
def missingData(df):
    df.select([count(when(col(c).isNull(),lit(1))).alias(c) for c in df.columns]).show()

In [21]:
missingData(customer_df)
missingData(geolocation_df)
missingData(order_items_df)
missingData(order_payments_df)
missingData(order_reviews_df)
missingData(orders_df)
missingData(products_df)
missingData(sellers_df)
missingData(product_category_name_df)


+-----------+------------------+------------------------+-------------+--------------+
|customer_id|customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|
+-----------+------------------+------------------------+-------------+--------------+
|          0|                 0|                       0|            0|             0|
+-----------+------------------+------------------------+-------------+--------------+



+---------------------------+---------------+---------------+----------------+-----------------+
|geolocation_zip_code_prefix|geolocation_lat|geolocation_lng|geolocation_city|geolocation_state|
+---------------------------+---------------+---------------+----------------+-----------------+
|                          0|              0|              0|               0|                0|
+---------------------------+---------------+---------------+----------------+-----------------+



+--------+-------------+----------+---------+-------------------+-----+-------------+
|order_id|order_item_id|product_id|seller_id|shipping_limit_date|price|freight_value|
+--------+-------------+----------+---------+-------------------+-----+-------------+
|       0|            0|         0|        0|                  0|    0|            0|
+--------+-------------+----------+---------+-------------------+-----+-------------+

+--------+------------------+------------+--------------------+-------------+
|order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------+------------------+------------+--------------------+-------------+
|       0|                 0|           0|                   0|            0|
+--------+------------------+------------+--------------------+-------------+



+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+
|review_id|order_id|review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+
|        1|    2236|        2380|               92157|                 63079|                8764|                   8785|
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+



+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|       0|          0|           0|                       0|              160|                        1783|                         2965|                            0|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+

+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+------

### JOINING ALL THE TABLES

In [9]:
order_item_joined_df= orders_df.join(order_items_df, 'order_id','inner')

In [80]:
order_item_joined_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)



In [10]:
order_item_products_df =    order_item_joined_df.join(products_df,'product_id','inner')

In [82]:
order_item_products_df.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (null

In [11]:
order_item_products_sellers_df = order_item_products_df.join(broadcast(sellers_df), 'seller_id','inner')

In [84]:
order_item_products_sellers_df.printSchema()

root
 |-- seller_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (null

In [12]:
full_orders_df= order_item_products_sellers_df.join(customer_df, 'customer_id', 'inner')

In [86]:
full_orders_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (null

In [13]:
full_orders_df= full_orders_df.join(broadcast(geolocation_df), full_orders_df.customer_zip_code_prefix == geolocation_df.geolocation_zip_code_prefix,'left')

In [93]:
full_orders_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (null

In [14]:
full_orders_df= full_orders_df.join(broadcast(order_reviews_df), 'order_id' ,'left')

In [15]:
full_orders_df= full_orders_df.join(order_payments_df,'order_id','left')

In [16]:
full_orders_df.cache()

25/10/02 18:03:34 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


DataFrame[order_id: string, customer_id: string, seller_id: string, product_id: string, order_status: string, order_purchase_timestamp: timestamp, order_approved_at: timestamp, order_delivered_carrier_date: timestamp, order_delivered_customer_date: timestamp, order_estimated_delivery_date: timestamp, order_item_id: int, shipping_limit_date: timestamp, price: double, freight_value: double, product_category_name: string, product_name_lenght: int, product_description_lenght: int, product_photos_qty: int, product_weight_g: int, product_length_cm: int, product_height_cm: int, product_width_cm: int, seller_zip_code_prefix: int, seller_city: string, seller_state: string, customer_unique_id: string, customer_zip_code_prefix: int, customer_city: string, customer_state: string, geolocation_zip_code_prefix: int, geolocation_lat: double, geolocation_lng: double, geolocation_city: string, geolocation_state: string, review_id: string, review_score: string, review_comment_title: string, review_commen

In [14]:
full_orders_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (null

In [15]:
total_revenue= full_orders_df.groupBy('seller_id').agg(_sum('price').alias('total_revenue'))

In [16]:
total_revenue.orderBy('total_revenue', ascending=False).show()


+--------------------+--------------------+
|           seller_id|       total_revenue|
+--------------------+--------------------+
|4869f7a5dfa277a7d...| 3.613871731999314E7|
|53243585a1d6dc264...|3.4291592950000696E7|
|4a3ca9315b744ce9f...| 3.375957084001202E7|
|7c67e1448b00f6e96...|3.2282321790021457E7|
|fa1c13f2614d7b5c4...|3.0139386310006626E7|
|da8622b14eb17ae28...| 2.985766973003611E7|
|7e93a43ef30c4f03f...| 2.631570630000355E7|
|1025f0e2d44d7041d...|2.2937518520012792E7|
|46dc3b2cc0980fb8e...| 2.179177329001608E7|
|955fee9216a65b617...|2.0964410670017645E7|
|7a67c85e85bb2ce85...| 2.031279489002914E7|
|620c87c171fb2a6dd...| 2.011983960002505E7|
|7d13fca1522535862...|1.8156881910003982E7|
|a1043bafd471dff53...|1.7662675980010696E7|
|6560211a19b47992c...|1.7315932900000528E7|
|edb1ef5e36e0c8cd8...|1.6624835150007125E7|
|1f50f920176fa81da...|1.6497454440033637E7|
|5dceca129747e92ff...|1.4910548340003535E7|
|cc419e0650a3c5ba7...| 1.475146450003979E7|
|3d871de0142ce09b7...|1.41845253

In [17]:
#-----tasks----
#Total order per customer
#avg review score for selller
#most sold product(top 10)
#top customers by spending


In [17]:
#Total order per customer

total_order_per_customer = full_orders_df.groupBy("customer_id") \
    .agg(count(col("order_id")).alias("total_orders"))



In [18]:
total_order_per_customer.orderBy('total_orders', ascending=False).show()

+--------------------+------------+
|         customer_id|total_orders|
+--------------------+------------+
|351e40989da90e704...|       11427|
|50920f8cd0681fd86...|       10752|
|9b43e2a62de9bab3a...|        8556|
|270c23a11d024a44c...|        8001|
|5c87184371002d49e...|        6876|
|d3e82ccec3cb5f956...|        6876|
|d5f2b3f597c7ccafb...|        6706|
|c2f18647725395af4...|        6612|
|24e7dc2ff8c071263...|        6597|
|7bb57d182bdc11653...|        6258|
|63b964e79dee32a35...|        6072|
|d22f25a9fadfb1abb...|        6072|
|1ff773612ab8934db...|        5820|
|13aa59158da63ba0e...|        5206|
|78fc46047c4a639e8...|        5200|
|dd3f1762eb601f41c...|        4992|
|a193aa8d905b8e246...|        4896|
|9eb3d566e87289dcb...|        4872|
|2ba91e12e5e4c9f56...|        4752|
|55e7cfd6e28d2fbfb...|        4728|
+--------------------+------------+
only showing top 20 rows



In [19]:
#avg review score for selller

avg_review_score_for_seller = full_orders_df.groupBy("seller_id") \
    .agg(
        avg("review_score").alias("avg_review_score"),
        count("review_score").alias("num_reviews")
    )



In [20]:
avg_review_score_for_seller.orderBy("avg_review_score", ascending=False).show()


+--------------------+----------------+-----------+
|           seller_id|avg_review_score|num_reviews|
+--------------------+----------------+-----------+
|58c851d1a3c7cd3da...|             5.0|        191|
|aa8af66c623d7d544...|             5.0|        177|
|0f519b0d2e5eb2227...|             5.0|         38|
|31e60bf8d103ce479...|             5.0|          9|
|64c9a1db4e73e19aa...|             5.0|        210|
|2b2fed75b8e5ea3a0...|             5.0|        528|
|33ab10be054370c25...|             5.0|        849|
|9d213f303afae4983...|             5.0|        146|
|05a48cc8859962767...|             5.0|         54|
|94d76e96eedd97625...|             5.0|        241|
|e94b64dc6979b302a...|             5.0|        587|
|4aba6a02a788d3ec8...|             5.0|        220|
|fd312b6bf05efac6c...|             5.0|        428|
|a61cc04793308395a...|             5.0|        116|
|1a8e2d9c38b84a970...|             5.0|        203|
|edb58a1390adf2738...|             5.0|        124|
|c04d70d515d

In [24]:
#most sold product(top 10)

most_sold_product = full_orders_df.groupBy("product_id") \
    .agg(count(col("order_item_id")).alias("total_items_sold")) \
    .orderBy("total_items_sold", ascending=False)



In [25]:
most_sold_product.show(10)


+--------------------+----------------+
|          product_id|total_items_sold|
+--------------------+----------------+
|aca2eb7d00ea1a7b8...|           86740|
|422879e10f4668299...|           81110|
|99a4788cb24856965...|           78775|
|389d119b48cf3043d...|           60248|
|d1c427060a0f73f6b...|           59274|
|368c6c730842d7801...|           58358|
|53759a2ecddad2bb8...|           52654|
|53b36df67ebb7c415...|           52105|
|154e7e31ebfa09220...|           42700|
|3dd2a17168ec895c7...|           40787|
+--------------------+----------------+
only showing top 10 rows



In [26]:
#top customers by spending


# Step 1: Aggregate payment per order
order_payments_total = order_payments_df.groupBy("order_id") \
    .agg(_sum("payment_value").alias("order_payment_total"))

# Step 2: Map order -> customer (deduplicate orders)
order_customer_map = full_orders_df.select("order_id", "customer_unique_id") \
    .dropDuplicates(["order_id"])

# Step 3: Join and compute total spend per customer
top_customers_by_spending = order_customer_map.join(order_payments_total, on="order_id", how="inner") \
    .groupBy("customer_unique_id") \
    .agg(_sum("order_payment_total").alias("total_spending")) \
    .orderBy("total_spending", ascending=False)

top_customers_by_spending.show(10, truncate=False)


+--------------------------------+--------------+
|customer_unique_id              |total_spending|
+--------------------------------+--------------+
|0a0a92112bd4c708ca5fde585afaa872|13664.08      |
|da122df9eeddfedc1dc1f5349a1a690c|7571.63       |
|763c8b1c9c68a0229c42c9fc6f662b93|7274.88       |
|dc4802a71eae9be1dd28f5d788ceb526|6929.31       |
|459bef486812aa25204be022145caa62|6922.21       |
|ff4159b92c40ebe40454e3e6a7c35ed6|6726.66       |
|4007669dec559734d6f53e029e360987|6081.54       |
|5d0a2980b292d049061542014e8960bf|4809.44       |
|eebb5dda148d3893cdaf5b5ca3040ccb|4764.34       |
|48e1ac109decbb87765a3eade6854098|4681.78       |
+--------------------------------+--------------+
only showing top 10 rows



In [27]:
#window functions and ranking

full_orders_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (null

In [28]:
#rank top selling products per seller

window_spec= Window.partitionBy('seller_id').orderBy(desc('price'))

top_selling_products= full_orders_df.withColumn('rank', rank ().over(window_spec))

In [43]:
top_selling_products.select('seller_id','price', 'rank').show()

+--------------------+-----+----+
|           seller_id|price|rank|
+--------------------+-----+----+
|0015a82c2db000af6...|895.0|   1|
|0015a82c2db000af6...|895.0|   1|
|0015a82c2db000af6...|895.0|   1|
|0015a82c2db000af6...|895.0|   1|
|0015a82c2db000af6...|895.0|   1|
|0015a82c2db000af6...|895.0|   1|
|0015a82c2db000af6...|895.0|   1|
|0015a82c2db000af6...|895.0|   1|
|0015a82c2db000af6...|895.0|   1|
|0015a82c2db000af6...|895.0|   1|
|0015a82c2db000af6...|895.0|   1|
|0015a82c2db000af6...|895.0|   1|
|0015a82c2db000af6...|895.0|   1|
|0015a82c2db000af6...|895.0|   1|
|0015a82c2db000af6...|895.0|   1|
|0015a82c2db000af6...|895.0|   1|
|0015a82c2db000af6...|895.0|   1|
|0015a82c2db000af6...|895.0|   1|
|0015a82c2db000af6...|895.0|   1|
|0015a82c2db000af6...|895.0|   1|
+--------------------+-----+----+
only showing top 20 rows



In [46]:
#dense rank for seller based on revenue

In [30]:
seller_revenue= full_orders_df.groupBy('seller_id').agg(_sum(col('payment_value')).alias('total_revenue'))

In [31]:
window_spec= Window.orderBy(desc('total_revenue'))

seller_based_revenue = seller_revenue.withColumn('dense_rank', dense_rank().over(window_spec))        

seller_based_revenue.show(5)

25/10/02 14:00:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/02 14:00:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/02 14:00:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/02 14:00:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/02 14:00:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/02 14:00:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/02 1

+--------------------+-------------------+----------+
|           seller_id|      total_revenue|dense_rank|
+--------------------+-------------------+----------+
|7c67e1448b00f6e96...|8.568359756000341E7|         1|
|da8622b14eb17ae28...|5.381143432998673E7|         2|
|1025f0e2d44d7041d...|5.043467467999159E7|         3|
|4a3ca9315b744ce9f...|4.872729552001618E7|         4|
|1f50f920176fa81da...|4.457027814000921E7|         5|
+--------------------+-------------------+----------+
only showing top 5 rows



In [32]:
#dense rank for seller based on revenue
window_spec= Window.partitionBy('seller_id').orderBy(desc('payment_value'))

top_seller_revenue= full_orders_df.withColumn('dense_rank', dense_rank().over(window_spec))
top_seller_revenue.select('seller_id', 'payment_value', 'dense_rank').show(5)


+--------------------+-------------+----------+
|           seller_id|payment_value|dense_rank|
+--------------------+-------------+----------+
|0015a82c2db000af6...|       916.02|         1|
|0015a82c2db000af6...|       916.02|         1|
|0015a82c2db000af6...|       916.02|         1|
|0015a82c2db000af6...|       916.02|         1|
|0015a82c2db000af6...|       916.02|         1|
+--------------------+-------------+----------+
only showing top 5 rows



In [33]:
top_seller_revenue.select('seller_id', 'payment_value', 'dense_rank').show(5)

+--------------------+-------------+----------+
|           seller_id|payment_value|dense_rank|
+--------------------+-------------+----------+
|0015a82c2db000af6...|       916.02|         1|
|0015a82c2db000af6...|       916.02|         1|
|0015a82c2db000af6...|       916.02|         1|
|0015a82c2db000af6...|       916.02|         1|
|0015a82c2db000af6...|       916.02|         1|
+--------------------+-------------+----------+
only showing top 5 rows



In [68]:
#advance aggration and enrichment

In [69]:
# total revenue and avg order value(AOV) per customer

In [96]:
customer_spending_df= full_orders_df.groupBy('customer_id').agg(count('order_id').alias('total_orders'),
                                                                _sum('price').alias('total_spend'),
                                                                _round(avg('price'),2).alias('AOV'))\
                                    .orderBy(desc('total_spend'))


In [ ]:
customer_spending_df.show()

In [75]:
#seller performance metrics(revenue, avg review, order count)

In [35]:
seller_performance = full_orders_df.groupBy('seller_id').agg(count('order_id').alias('total_order'),
                                                             _sum('price').alias('total_revenue'),
                                                             _round(avg('review_score'),2).alias('avg_review_score'),
                                                             _round(stddev('price'),2).alias('price variability'))\
                                    .orderBy(desc('total_revenue')).show()

+--------------------+-----------+--------------------+----------------+-----------------+
|           seller_id|total_order|       total_revenue|avg_review_score|price variability|
+--------------------+-----------+--------------------+----------------+-----------------+
|4869f7a5dfa277a7d...|     184587| 3.613871731999314E7|            4.09|           111.65|
|53243585a1d6dc264...|      54514|3.4291592950000696E7|            4.12|           499.65|
|4a3ca9315b744ce9f...|     330661| 3.375957084001202E7|            3.77|            59.37|
|7c67e1448b00f6e96...|     233306|3.2282321790021457E7|            3.42|            50.39|
|fa1c13f2614d7b5c4...|      87686|3.0139386310006626E7|            4.38|            307.7|
|da8622b14eb17ae28...|     264433| 2.985766973003611E7|            3.98|            72.92|
|7e93a43ef30c4f03f...|      50226| 2.631570630000355E7|            4.15|           377.24|
|1025f0e2d44d7041d...|     229587|2.2937518520012792E7|            3.89|             84.3|

In [36]:
#product popularity Metrics

from pyspark.sql.functions import collect_list,collect_set

In [37]:
product_metrics_df= full_orders_df.groupBy('product_id').agg( count('order_id').alias('total_sales'),
                                                             _sum('price').alias('total_revenue'),
                                                             _round(avg('price'),2).alias('avg_price'),
                                                             _round(stddev('price'),2).alias('price variability'),
                                                             collect_list('seller_id').alias('seller_list'),
                                                             collect_set('seller_id').alias('unique seller'))\
                                    .orderBy(desc('total_sales')).show()

+--------------------+-----------+------------------+---------+-----------------+--------------------+--------------------+
|          product_id|total_sales|     total_revenue|avg_price|price variability|         seller_list|       unique seller|
+--------------------+-----------+------------------+---------+-----------------+--------------------+--------------------+
|aca2eb7d00ea1a7b8...|      86740| 6164630.299996104|    71.07|             3.17|[955fee9216a65b61...|[955fee9216a65b61...|
|422879e10f4668299...|      81110| 4442791.509997985|    54.77|             4.46|[1f50f920176fa81d...|[1f50f920176fa81d...|
|99a4788cb24856965...|      78775| 6921762.709996391|    87.87|             4.08|[4a3ca9315b744ce9...|[4a3ca9315b744ce9...|
|389d119b48cf3043d...|      60248| 3280533.129998789|    54.45|             4.37|[1f50f920176fa81d...|[1f50f920176fa81d...|
|d1c427060a0f73f6b...|      59274| 8220103.330003063|   138.68|            16.58|[a1043bafd471dff5...|[a1043bafd471dff5...|
|368c6c7

In [87]:
full_orders_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (null

In [38]:
from pyspark.sql.functions import month,min as _min,max as _max

In [39]:
#monthly revenue and order count trends -task

#total_order, total_revenue, avg order_value, min_order_value,max_order_value
#groupBy--> order purchase timestamp  ---> extract month
#price = seller product value → good for seller revenue analysis.
#payment_value = customer payment (includes shipping, etc.) → good for customer spend analysis.

full_order_month_extracted= full_orders_df.withColumn('order_purchase_month' , month(col('order_purchase_timestamp')))


In [43]:
monthly_revenue= full_order_month_extracted.groupBy('order_purchase_month').agg( count('order_id').alias('total_orders'),
                                                                                _round(_sum('price'),5).alias('total_revenue'),
                                                                                _round(avg('price'),2).alias('avg_order_value'),
                                                                                _round(_min('price'),2).alias('min_order_value'),
                                                                                _round(_max('price'),2).alias('max_order_value'))\
                                            .orderBy(desc('total_orders'))

#converted total_revenue from scientific notation(2.4006115197E8 to 240061151.97) using .cast (.cast(DecimalType(20,2))
monthly_revenue = monthly_revenue.withColumn("total_revenue", col("total_revenue").cast(DecimalType(20,2)))
monthly_revenue.show(truncate=False)




+--------------------+------------+-------------+---------------+---------------+---------------+
|order_purchase_month|total_orders|total_revenue|avg_order_value|min_order_value|max_order_value|
+--------------------+------------+-------------+---------------+---------------+---------------+
|5                   |1918571     |240061151.97 |125.12         |3.5            |6499.0         |
|8                   |1914313     |231958707.85 |121.17         |2.2            |4399.87        |
|7                   |1847639     |222908857.10 |120.65         |1.2            |6729.0         |
|3                   |1809467     |218681168.43 |120.85         |4.9            |4099.99        |
|6                   |1701909     |210243323.49 |123.53         |3.49           |4590.0         |
|4                   |1693860     |217156969.13 |128.2          |0.85           |4799.0         |
|2                   |1551163     |178781784.07 |115.26         |2.99           |6735.0         |
|1                  

In [48]:
# customer retention analysis (frist and last order)
#unique_orders = full_orders_df.select("customer_id", "order_id", "order_purchase_timestamp").distinct()


unique_orders = full_orders_df.select(
    "customer_id", "order_id", "order_purchase_timestamp"
).distinct()

customer_retention_df= unique_orders.groupBy('customer_id').agg( _min('order_purchase_timestamp').alias('first_order_date'),
                                                                 _max('order_purchase_timestamp').alias('last_order_date'),
                                                                 count('order_id').alias('total_orders'))\
                                    .orderBy(desc('total_orders'))

customer_retention_df.show()
                                        



+--------------------+-------------------+-------------------+------------+
|         customer_id|   first_order_date|    last_order_date|total_orders|
+--------------------+-------------------+-------------------+------------+
|9e67b6474f7e8370b...|2017-07-03 09:28:22|2017-07-03 09:28:22|           1|
|b6b70c77dbd33d427...|2017-06-26 09:29:20|2017-06-26 09:29:20|           1|
|843a771c6235fdb0a...|2018-06-15 15:23:07|2018-06-15 15:23:07|           1|
|1c098ad91b1234eaf...|2017-08-17 09:41:02|2017-08-17 09:41:02|           1|
|c61b59e94df1163ca...|2018-03-06 13:22:10|2018-03-06 13:22:10|           1|
|a6aa137231a82a6c8...|2018-04-14 17:51:24|2018-04-14 17:51:24|           1|
|602cd37236aba223c...|2017-06-13 12:32:12|2017-06-13 12:32:12|           1|
|6a5e444906153f324...|2017-05-05 08:44:57|2017-05-05 08:44:57|           1|
|2a8da2a879305eff0...|2018-03-26 00:13:51|2018-03-26 00:13:51|           1|
|50fc555de6aad7ded...|2018-04-07 20:22:01|2018-04-07 20:22:01|           1|
|f058edb3bbb

In [50]:
customer_retention_df= customer_retention_df.withColumn(
        "retention_days",
        date_diff(col("last_order_date"), col("first_order_date")))\
    .orderBy(desc("total_orders"))

In [51]:
customer_retention_df.show()


+--------------------+-------------------+-------------------+------------+--------------+
|         customer_id|   first_order_date|    last_order_date|total_orders|retention_days|
+--------------------+-------------------+-------------------+------------+--------------+
|27d900ed012d6b80d...|2017-10-01 10:33:50|2017-10-01 10:33:50|           1|             0|
|bc6063f0f2cde26be...|2018-06-10 21:23:13|2018-06-10 21:23:13|           1|             0|
|874b661d62c1e74aa...|2017-11-20 21:53:20|2017-11-20 21:53:20|           1|             0|
|52f7baf30ea546558...|2017-05-11 13:22:02|2017-05-11 13:22:02|           1|             0|
|ef019b3cc077faf03...|2018-06-06 21:47:57|2018-06-06 21:47:57|           1|             0|
|42da09831872a4ecc...|2018-07-25 21:29:05|2018-07-25 21:29:05|           1|             0|
|b9c6a2e20686e1df4...|2018-01-26 11:00:07|2018-01-26 11:00:07|           1|             0|
|ac9b518157bd24e32...|2017-10-23 19:29:00|2017-10-23 19:29:00|           1|             0|

In [54]:
customer_retention_df.filter(col("total_orders") > 1).orderBy(desc("retention_days")).show(10, truncate=False)


+-----------+----------------+---------------+------------+--------------+
|customer_id|first_order_date|last_order_date|total_orders|retention_days|
+-----------+----------------+---------------+------------+--------------+
+-----------+----------------+---------------+------------+--------------+



In [59]:
full_orders_df.printSchema()

root
 |-- order_status: string (nullable = true)



In [74]:
#advance enrichment
#order status flags

full_orders_filter_df= full_orders_df.select('order_id','order_status')

In [87]:
full_orders_df_flags= full_orders_filter_df.withColumn('is_delivered',  when((col('order_status')=='delivered'),lit(1)).otherwise(lit(0)))\
                                    .withColumn('is_cancelled', when((col('order_status')=='canceled'),lit(1)).otherwise(lit(0)))
                         
                         
full_orders_df_flags.where(col('is_cancelled')==lit(1)).show()
                         

+--------------------+------------+------------+------------+
|            order_id|order_status|is_delivered|is_cancelled|
+--------------------+------------+------------+------------+
|00310b0c75bb13015...|    canceled|           0|           1|
|00310b0c75bb13015...|    canceled|           0|           1|
|00310b0c75bb13015...|    canceled|           0|           1|
|00310b0c75bb13015...|    canceled|           0|           1|
|00310b0c75bb13015...|    canceled|           0|           1|
|00310b0c75bb13015...|    canceled|           0|           1|
|00310b0c75bb13015...|    canceled|           0|           1|
|00310b0c75bb13015...|    canceled|           0|           1|
|00310b0c75bb13015...|    canceled|           0|           1|
|00310b0c75bb13015...|    canceled|           0|           1|
|00310b0c75bb13015...|    canceled|           0|           1|
|00310b0c75bb13015...|    canceled|           0|           1|
|00310b0c75bb13015...|    canceled|           0|           1|
|00310b0

In [88]:

flag_summary = full_orders_df_flags.agg(
    _sum("is_delivered").alias("delivered_count"),
    _sum("is_cancelled").alias("cancelled_count")
)

flag_summary.show()


+---------------+---------------+
|delivered_count|cancelled_count|
+---------------+---------------+
|       17692295|          81273|
+---------------+---------------+



In [85]:

order_status_summary = full_orders_df_flags.groupBy("order_status").agg(
    count("*").alias("total_orders")
)

order_status_summary.show()


+------------+------------+
|order_status|total_orders|
+------------+------------+
|   delivered|    17692295|
|    canceled|       81273|
|     shipped|      165508|
|    approved|         658|
|  processing|       59073|
|    invoiced|       64213|
| unavailable|        1241|
+------------+------------+



In [90]:
full_orders_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (null

In [89]:
#order_revenue_calculation 

In [91]:
order_revenue_calculation= full_orders_df.withColumn('order_revenue', col('price')+col('freight_value'))

In [93]:
order_revenue_calculation.select('price','freight_value','order_revenue').show(10)

+-----+-------------+-------------+
|price|freight_value|order_revenue|
+-----+-------------+-------------+
| 58.9|        13.29|        72.19|
| 58.9|        13.29|        72.19|
| 58.9|        13.29|        72.19|
| 58.9|        13.29|        72.19|
| 58.9|        13.29|        72.19|
| 58.9|        13.29|        72.19|
| 58.9|        13.29|        72.19|
| 58.9|        13.29|        72.19|
| 58.9|        13.29|        72.19|
| 58.9|        13.29|        72.19|
+-----+-------------+-------------+
only showing top 10 rows



In [97]:
customer_spending_df.show(10)

+--------------------+------------+------------------+-------+
|         customer_id|total_orders|       total_spend|    AOV|
+--------------------+------------+------------------+-------+
|d3e82ccec3cb5f956...|        6876|         6662844.0|  969.0|
|df55c14d1476a9a34...|         743|         3565657.0| 4799.0|
|fe5113a38e3575c04...|        2292|         3293604.0| 1437.0|
|ec5b2ba62e5743423...|        1428|         2556120.0| 1790.0|
|63b964e79dee32a35...|        6072|         2501664.0|  412.0|
|46bb3c0b1a65c8399...|         748|         2336752.0| 3124.0|
|05455dfa7cd02f13d...|        2184| 2160194.400000087|  989.1|
|3690e975641f01bd0...|         802|         2124498.0| 2649.0|
|349509b216bd5ec11...|         743|         1923627.0| 2589.0|
|695476b5848d64ba0...|         687|1820543.1299999943|2649.99|
+--------------------+------------+------------------+-------+
only showing top 10 rows



In [98]:
#customer segmentation based on spending

customer_spending_segmentation_df = customer_spending_df.withColumn('cutomer_segment',
                                                                    when((col('AOV')>1200),'High_value')\
                                                                    .when((col('AOV')>700) & (col('AOV')<=1200), 'Medium_value')\
                                                                    .otherwise('Low_value'))

In [102]:
customer_spending_segmentation_df.groupBy('cutomer_segment').agg(count('*').alias('total_customer')).show()

+---------------+--------------+
|cutomer_segment|total_customer|
+---------------+--------------+
|      Low_value|         97000|
|     High_value|           540|
|   Medium_value|          1126|
+---------------+--------------+



In [103]:
full_cutomer_segment_df= full_orders_df.join(customer_spending_segmentation_df.select('customer_id','cutomer_segment'), 'customer_id','left')

In [114]:
full_cutomer_segment_df.select('customer_id','cutomer_segment').where(col('cutomer_segment')=='High_value').show(5)

+--------------------+---------------+
|         customer_id|cutomer_segment|
+--------------------+---------------+
|44b701cb9a97bd178...|     High_value|
|44b701cb9a97bd178...|     High_value|
|44b701cb9a97bd178...|     High_value|
|44b701cb9a97bd178...|     High_value|
|44b701cb9a97bd178...|     High_value|
+--------------------+---------------+
only showing top 5 rows



In [125]:
#hourly order distribution

from pyspark.sql.functions import expr, dayofweek

In [119]:
full_orders_hourly_df= full_orders_df.withColumn('hourOfDay', expr('hour(order_purchase_timestamp)'))

In [120]:
full_orders_hourly_df.select('hourOfDay', 'order_purchase_timestamp').show()

+---------+------------------------+
|hourOfDay|order_purchase_timestamp|
+---------+------------------------+
|        8|     2017-09-13 08:59:02|
|        8|     2017-09-13 08:59:02|
|        8|     2017-09-13 08:59:02|
|        8|     2017-09-13 08:59:02|
|        8|     2017-09-13 08:59:02|
|        8|     2017-09-13 08:59:02|
|        8|     2017-09-13 08:59:02|
|        8|     2017-09-13 08:59:02|
|        8|     2017-09-13 08:59:02|
|        8|     2017-09-13 08:59:02|
|        8|     2017-09-13 08:59:02|
|        8|     2017-09-13 08:59:02|
|        8|     2017-09-13 08:59:02|
|        8|     2017-09-13 08:59:02|
|        8|     2017-09-13 08:59:02|
|        8|     2017-09-13 08:59:02|
|        8|     2017-09-13 08:59:02|
|        8|     2017-09-13 08:59:02|
|        8|     2017-09-13 08:59:02|
|        8|     2017-09-13 08:59:02|
+---------+------------------------+
only showing top 20 rows



In [121]:
#weekday or weekend orders


In [149]:
full_orders_hourly_df= full_orders_df.withColumn(
    'order_day_type', when(dayofweek(col('order_purchase_timestamp')).isin([1,7]),'Weekend')
    .otherwise('Weekday')).join(full_cutomer_segment_df, 'customer_id','left') #used join to check the cutomer_segment(High_value)                                                                              

In [153]:
full_orders_hourly_df.filter((col('order_day_type')=='Weekday') & (col('cutomer_segment')=='Low_value'))\
.select('customer_id','order_day_type', 'cutomer_segment').show(10)


+--------------------+--------------+---------------+
|         customer_id|order_day_type|cutomer_segment|
+--------------------+--------------+---------------+
|000419c5494106c30...|       Weekday|      Low_value|
|000419c5494106c30...|       Weekday|      Low_value|
|000419c5494106c30...|       Weekday|      Low_value|
|000419c5494106c30...|       Weekday|      Low_value|
|000419c5494106c30...|       Weekday|      Low_value|
|000419c5494106c30...|       Weekday|      Low_value|
|000419c5494106c30...|       Weekday|      Low_value|
|000419c5494106c30...|       Weekday|      Low_value|
|000419c5494106c30...|       Weekday|      Low_value|
|000419c5494106c30...|       Weekday|      Low_value|
+--------------------+--------------+---------------+
only showing top 10 rows



In [156]:
full_orders_df.select('freight_value').show(30)

+-------------+
|freight_value|
+-------------+
|        13.29|
|        13.29|
|        13.29|
|        13.29|
|        13.29|
|        13.29|
|        13.29|
|        13.29|
|        13.29|
|        13.29|
|        13.29|
|        13.29|
|        13.29|
|        13.29|
|        13.29|
|        13.29|
|        13.29|
|        13.29|
|        13.29|
|        13.29|
|        13.29|
|        13.29|
|        13.29|
|        13.29|
|        13.29|
|        13.29|
|        13.29|
|        13.29|
|        13.29|
|        13.29|
+-------------+
only showing top 30 rows



In [160]:
#1-(freight_category- low, ,medium, high)

freight_category_df= full_orders_df.select('customer_id','freight_value')\
                        .withColumn('freight_category', when ((col('freight_value')>10),lit('High_freight'))\
                                    .when((col('freight_value')<=10) & (col('freight_value')>4) ,lit('Midium_freight'))\
                                    .otherwise('Low_freight'))

In [161]:
full_freight_category_df= full_orders_df.join(freight_category_df, 'customer_id', 'left')

In [164]:
full_orders_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (null

In [178]:
#2 order volume by customer state

orderVolume_customerState_df= full_orders_df.groupBy('customer_state').agg(count('order_id').alias("order_count"))
orderVolume_customerState_df.show()

+--------------+-----------+
|customer_state|order_count|
+--------------+-----------+
|            MS|      73693|
|            CE|      74749|
|            MG|    3433239|
|            DF|     109466|
|            RO|      24523|
|            AM|       6488|
|            MT|     155233|
|            SP|    6742255|
|            PB|      33381|
|            BA|     443992|
|            SE|      28146|
|            RJ|    3626875|
|            AC|       8686|
|            PR|     746540|
|            AP|       5698|
|            RR|       2411|
|            TO|      22361|
|            ES|     367217|
|            AL|      37742|
|            RN|      24820|
+--------------+-----------+
only showing top 20 rows



In [183]:
#3- save as parquet

orderVolume_customerState_df.write.mode('overwrite').parquet('/tmp/orderVolume_customerState_df')

In [187]:
!hadoop fs -ls /tmp

Found 5 items
drwxr-xr-x   - ankiitsharma09 hadoop          0 2025-09-02 17:37 /tmp/10MB
drwxr-xr-x   - ankiitsharma09 hadoop          0 2025-09-12 04:30 /tmp/150MB
drwxr-xr-x   - ankiitsharma09 hadoop          0 2025-09-02 17:36 /tmp/1MB
drwxrwxrwt   - hdfs           hadoop          0 2025-08-16 18:02 /tmp/hadoop-yarn
drwx-wx-wx   - hive           hadoop          0 2025-08-16 18:02 /tmp/hive


## Module 4 - Spark configuration and optimization

In [1]:
spark.stop()


In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder\
    .appName("E-Commerce Order Analytics")\
    .master("yarn")\
    .config("spark.executor.instances", 2)\
    .config("spark.executor.cores", 2)\
    .config("spark.executor.memory", "3g")\
    .config("spark.driver.memory", "4g")\
    .getOrCreate()


25/10/02 17:55:26 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [ ]:
spark = SparkSession.builder\
    .appName('spark_configuration')\
    .master("yarn")\
    .config('spark.executor.cores', 2)\
    .config('spark.executor.instances', 2)\
    .config('spark.driver.memory', '4g')\
    .config('spark.driver.maxResultSize', '2g')\
    .config('spark.sql.shuffle.partitions', 64)\
    .config('spark.default.parallelism', 64)\
    .config('spark.sql.adaptive.enabled', True)\
    .config('spark.sql.adaptive.coalescePartitions.enabled', True)\
    .config('spark.sql.autoBroadcastJoinThreshold', 20 * 1024 * 1024)\
    .config('spark.sql.files.maxPartitionBytes', '64MB')\
    .config('spark.files.openCostInBytes', '2MB')\
    .config('spark.memory.fraction', 0.8)\
    .config('spark.memory.storageFraction', 0.2)\
    .getOrCreate()


In [2]:
#join optimization strategies

In [18]:
#broadcast join
customer_broadcast_df= broadcast(customer_df)
optimized_broadcast_df= full_orders_df.join(customer_broadcast_df, 'customer_id','inner')


In [21]:
#sort and merge join

sorted_customer_df= customer_df.sortWithinPartitions('customer_id')
sorted_orders_df= full_orders_df.sortWithinPartitions('customer_id')

optimized_merge_full_orders_df = sorted_orders_df.join(sorted_customer_df,'customer_id')

In [22]:
# Bucket join
bucketed_customers_df = customer_df.repartition(10,'customer_id')
bucketed_orders_df = full_orders_df.repartition(10,'customer_id')
bucket_join_df = bucketed_orders_df.join(bucketed_customers_df,'customer_id')

In [24]:
# Skew Join handling
skew_handled_join = full_orders_df.join(customer_df,'customer_id')

In [26]:
bucket_join_df.rdd.getNumPartitions()

10

In [30]:
!hadoop fs -rmdir /user/ankiitsharma09/ankiit/processedData

In [31]:
## Module 5 - save and retrive processed data

#save as prequet in hdfs

full_orders_df.write.mode('overwrite').parquet('/user/ankiitsharma09/ankiit/processedData')

In [32]:
#save as parquet in google cloud
full_orders_df.write.mode('overwrite').parquet('dataproc-temp-us-central1-854493138490-ruoapnvw')

In [34]:
#save as table
full_orders_df.write.mode('overwrite').saveAsTable('full_orders_processed')

DataFrame[namespace: string, tableName: string, isTemporary: boolean]

In [37]:
spark.sql('show tables').show(truncate=False)

+---------+---------------------+-----------+
|namespace|tableName            |isTemporary|
+---------+---------------------+-----------+
|default  |customer             |false      |
|default  |customer150mb        |false      |
|default  |full_orders_processed|false      |
|default  |revenue_per_seller   |false      |
+---------+---------------------+-----------+



In [ ]:
#save as csv
full_orders_df.write.mode('overwrite').option('header','true').csv('/user/ankiitsharma09/ankiit/processedData')


In [ ]:
#Project complete

In [38]:
spark.stop()